# Trajectory Analysis across models for _Suo et. al._ with Bootstapped Subgraphs

In [1]:
0

0

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad

sc.settings.verbose = 3

In [4]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.input import InputTrajectories
from sctram.generate.real import sc_suo_developmental_complete

2025-03-06 13:50:33.178 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /home/icb/kemal.inecik/work/codes/sctram/sctram/api/_defaults.yaml
2025-03-06 13:50:33.179 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [5]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

Download the dataset and preliminary subsetting

In [6]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [7]:
adata_suo_complete = sc_suo_developmental_complete(dataset_dir=dataset_dir)
display(adata_suo_complete)

2025-03-06 13:50:37.958 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/suo_developmental_complete.h5ad') already exists. Skipping download.


AnnData object with n_obs × n_vars = 841922 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', '_scvi_batch', '_scvi_labels', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'rank_genes_groups'
    obsm: 'Unintegrated', 'X_pca', 'harmony', 'invae', 'scanvi', 'scvi', 'tardis_1', 'tardis_2'

# Running the `sctram` package

In [22]:
input_trajectories_path = os.path.join(dataset_dir, f"adata_suo_input_haematopoeitic_lineage.pkl")
with open(input_trajectories_path, "rb") as _file:
    lit = pickle.load(_file)
little = lit.get_trajectory('haematopoeitic_lineage', False)
display(lit)
display(little)

InputTrajectories (trajectories=27, nodes=69, edges=297)

InputTrajectory (trajectory='haematopoeitic_lineage', nodes=67, edges=66)

In [23]:
litc_1 = InputTrajectories()
for trajectory in little.decompose_trajectory_method_1():
    litc_1.add_trajectory(trajectory)

input_trajectories_path_litc_1 = os.path.join(dataset_dir, f"adata_suo_input_haematopoeitic_lineage_litc_1.pkl")
with open(input_trajectories_path_litc_1, 'wb') as _file:
    pickle.dump(litc_1, _file)

len(litc_1.graph["trajectories"])

27

In [24]:
litc_2 = InputTrajectories()
for trajectory in little.decompose_trajectory_method_2():
    litc_2.add_trajectory(trajectory)

input_trajectories_path_litc_2 = os.path.join(dataset_dir, f"adata_suo_input_haematopoeitic_lineage_litc_2.pkl")
with open(input_trajectories_path_litc_2, 'wb') as _file:
    pickle.dump(litc_2, _file)

len(litc_2.graph["trajectories"])

114

In [25]:
override = False

count = 0
lineage = "Haematopoeitic_lineage"
lineage_part = lineage.replace("_lineage", "").lower()

for decompose_method in ["1", "2"]:

    input_trajectories_path_litc = os.path.join(dataset_dir, f"adata_suo_input_haematopoeitic_lineage_litc_{decompose_method}.pkl")
    with open(input_trajectories_path_litc, "rb") as _file:
        litc = pickle.load(_file)
    
    for use_rep in adata_suo_complete.obsm.keys():
        if use_rep == "Unintegrated":
            continue

        for trajectory in sorted(litc.graph["trajectories"]):

            output_file = os.path.join(dataset_dir, f"_metric_adata_suo_bootstrap_method_{decompose_method}_{lineage_part}_{use_rep}_{trajectory}.pickle")
            log_file = os.path.join(logs_directory, f"slurm_out_metric_adata_suo_bootstrap_method_{decompose_method}_{lineage_part}_{use_rep}_{trajectory}.log")
            slurmjob_name = f"sctram_bootstrap_{decompose_method}"
            
            if override or not os.path.exists(output_file) or not os.path.isfile(output_file):
                try:
                    slurm_script = f"""#!/bin/bash
#SBATCH -J {slurmjob_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=293G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, 'suo_sctram_bootstrap.py')} --lineage "{lineage}" --use_rep "{use_rep}" --decompose_method "{decompose_method}" --trajectory "{trajectory}"
    """
                    script_name = os.path.join(logs_directory, f"slurm_job_metric_adata_suo_bootstrap_method_{decompose_method}_{lineage_part}_{use_rep}_{trajectory}.sh")
                    with open(script_name, "w") as f:
                        f.write(slurm_script)
    
                    print(f"Submitted job {count+1!r} of {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for method {decompose_method!r}")
                    subprocess.run(["sbatch", script_name])
                    count += 1
                finally:
                    time.sleep(0.1)
                    os.remove(script_name)
            else:
                print(f"Data exists for {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for method {decompose_method!r}")
    # if count > 32:
    #     break

print(f" - Number of jobs submitted: {count}")

Submitted job 1 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lineage_subgraph_0' for method '1'
Submitted batch job 33748573
Submitted job 2 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lineage_subgraph_1' for method '1'
Submitted batch job 33748574
Submitted job 3 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lineage_subgraph_10' for method '1'
Submitted batch job 33748575
Submitted job 4 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lineage_subgraph_11' for method '1'
Submitted batch job 33748576
Submitted job 5 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lineage_subgraph_12' for method '1'
Submitted batch job 33748577
Submitted job 6 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lineage_subgraph_13' for method '1'
Submitted batch job 33748578
Submitted job 7 of 'Haematopoeitic_lineage' with 'X_pca' of trajectory 'haematopoeitic_lin

In [93]:
1

1